<h1 align="center">Física Computacional.</h1>
<h1 align="center">Semestre 2027-1</h1>

<h2>Sergio A. Alcalá Corona </h2> 

---
### Rodrigo Vega Vilchis
### Joel Gómez Uribe
---


<h1 align="center">Programación para la física computacional</h1> 

## Práctica 2.  Programación básica para la física

---

### Ejercicio 1. La constante de Madelung



En un cristal de NaCl (sal de mesa) los átomos de Na+ y Cl- forman una red cúbica alternada. La energía que siente un átomo en el origen es la suma del potencial de todos los demás átomos.

Si el átomo está en la posición $(i,j,k)$ a una distancia $\sqrt{(ia)^2+(ja)^2+(ka)^2}$, su potencial es:

$$ V(i,j,k) = \pm \frac{e}{4\pi\epsilon_0 a \sqrt{i^2+j^2+k^2}} $$

El signo es $+$ si $i+j+k$ es par (misma carga) y $-$ si es impar (carga opuesta).

El potencial total es $V_{total} = \frac{e}{4\pi\epsilon_0 a} M$, donde $M$ es la constante de Madelung.

**Instrucción oficial:** Escribir un programa que calcule e imprima la constante de Madelung $M$ para el cloruro de sodio. Técnicamente $M$ es cuando $L \to \infty$, pero se puede aproximar con un $L$ grande. Usar un $L$ lo más grande posible sin que el programa tarde más de 1 minuto en ejecutarse.

$$ M = \sum_{i,j,k=-L}^{L} \frac{\pm 1}{\sqrt{i^2+j^2+k^2}} $$

excluyendo $(0,0,0)$.

In [4]:
#bibliotecas
import numpy as np 
import matplotlib.pyplot as plt
import time
import math
import numba
###  Tipos de variables
L = 100  # int
M = 0.0 # float
print(f"L={L} tipo {type(L)}, M={M} tipo {type(M)}")

L=100 tipo <class 'int'>, M=0.0 tipo <class 'float'>


## Método 1: utilizando for

In [5]:
def madelung(L):# se define la funcion con dato L
    M = 0.0 #inicio donde se acumla la suma empieza en 0.0 
    for i in range(-L, L+1): 
        for j in range(-L, L+1):
            for k in range(-L, L+1):
                if i == 0 and j == 0 and k == 0:
                    continue   #= aquí no se hacen las operaciones matemáticas, se continúa a la siguiente vuelta.
                               #Si es diferente de 0 = aquí NO continuamos (no te saltes), aquí si se hace la suma
                r = math.sqrt(i*i + j*j + k*k) # distancia al origen
                if (i + j + k) % 2 == 0: # decide si el átomo es amigo o enemigo, si suma o resta.
                                         ## SI la suma i+j+k dividida entre 2 sobra 0 (o sea es PAR) ENTONCES:
                    M -= 1.0 / r         ## es del mismo tipo que el origen, resta,toma lo que ya tenías en M, réstale 1/r y guárdalo de nuevo en M
                else:
                    M += 1.0 / r        #si no (sobra 1, es IMPAR) es del tipo opuesto, suma
                                         ## toma lo que ya tenías en M, sumale 1/r y guárdalo de nuevo en M
    return M

# Aquí medimos tiempo  que tarda el programa en ejecutar cada L 
for L in [10, 50, 100, 150, 200]:
    inicio = time.time()
    resultado = madelung(L)
    fin = time.time()
    print(f"L={L} -> M={resultado:.8f} | Tardó {fin - inicio:.3f} segundos")

L=10 -> M=1.69257893 | Tardó 0.004 segundos
L=50 -> M=1.73613192 | Tardó 0.530 segundos
L=100 -> M=1.74181982 | Tardó 3.756 segundos
L=150 -> M=1.74372838 | Tardó 12.746 segundos
L=200 -> M=1.74468504 | Tardó 30.341 segundos


## Método 2: con numpy 

In [6]:
def madelung_rapido(L):
    M = 0.0
    vals = np.arange(-L, L+1)
    J, K = np.meshgrid(vals, vals, indexing='ij')
    for i in range(-L, L+1):
        R2 = i*i + J*J + K*K
        R2 = R2.astype(float)
        R2[R2==0] = np.inf
        R = np.sqrt(R2)
        signos = np.where((i + J + K) % 2 == 0, 1.0, -1.0)
        M += np.sum(signos / R)
    return M

for L in [10, 50, 100, 200]:
    inicio = time.time()
    Mm = madelung_rapido(L)
    print(f"L={L:3d} -> M={Mm:.6f} en {time.time()-inicio:.3f}s")

L= 10 -> M=-1.692579 en 0.002s
L= 50 -> M=-1.736132 en 0.025s
L=100 -> M=-1.741820 en 0.191s
L=200 -> M=-1.744685 en 3.422s


## metodo 3 : con numpy y numba

In [9]:
import numpy as np #para calcular np.sqrt
import numba # convierte triple for de p a c rapido
import time #para medir cuanto atarda L < 60 S

@numba.njit # TOMA LA FUNCION YA LA TRADUCE A LENGIAJE C
def mmadelung(L): #Def de la funcion de nombre mmadelung , variable L 
    M = 0.0 # ACUMULADOR M 0.0 FLOAT , sumammos divisiones 
    # solo octante positivo 0..L, no -L..L
    #  El ciclo exterior recorre i, el intermedio j y el interior k. 
    # Por cada combinación (i,j,k) se evalúa la distancia y si cumple la condición se acumula a la suma."
    for i in range(0, L+1): #"desde 0 hasta L" - el +1 es porque Python no cuenta el ultimo numero, 
        for j in range(0, L+1):
            for k in range(0, L+1):
                if i==0 and j==0 and k==0:   #si se cumplen estas comparacones , es decir si se esta en el origen no se cuenta y entonces se continua 
                    continue   ## Esto se salta el (0,0,0). 
                
                r = np.sqrt(i*i + j*j + k*k) # distancia al origen

                if (i + j + k) % 2 == 0: # suma i+j+k y se divide entre 2 , lo que sobre es el modulo, se valida si  es par o impar con el modulo , para saber si en na o cl 
                    s = -1.0  # par -> resta  #es del mismo tipo que el del origen Na con Na. Se repelen, por eso s = -1.0 (resta a la energía)
                else:
                    s = 1.0   # impar -> suma   #es del tipo opuesto (Na con Cl). Se atraen, por eso s = 1.0 (suma)

                 # truco de simetria: cuantas veces se repite este punto
                 #Como solo calculas el octante positivo i,j,k 0, cada punto que calculas en realidad existe 8 veces en el cubo completo
                mult = 1 #Inicializas el multiplicador en 1. Estás diciendo "este punto (i,j,k) por lo menos existe 1 vez, en mi octante
                if i != 0: mult *= 2 # ya tengo 2 copias por el eje x
                if j != 0: mult *= 2  # por cada copia en x, tengo 2 en y -> 2x2=4
                if k != 0: mult *= 2  # por cada copia en x,y, tengo 2 en z -> 2x2x2=8

                M += mult * s / r #A lo que ya llevo acumulado en M, se suma la contribución de este punto y todos sus 8 espejos: 
                #su signo dividido por su distancia, multiplicado por cuantas veces existe
    return M

# AQUI ponemos en que L queremos ver cuanto se tardo en calcular 
lista_L = [10, 100, 200, 500, 750, 1000, 2000]

for L in lista_L:
    inicio = time.time()
    Mm = mmadelung(L)
    duracion = time.time() - inicio
    print(f"L={L:4d} -> M={Mm:.6f} en {duracion:.3f}s")
    
    # si ya se pasó de 1 min, para para que no se quede pegado
    if duracion > 60:
        print("Se pasó de 1 min, parando.")
        break



L=  10 -> M=1.692579 en 0.503s
L= 100 -> M=1.741820 en 0.008s
L= 200 -> M=1.744685 en 0.061s
L= 500 -> M=1.746411 en 0.937s
L= 750 -> M=1.746795 en 3.164s
L=1000 -> M=1.746988 en 7.581s
L=2000 -> M=1.747276 en 59.590s


## **Ejercicio 2 - La formula semiempırica de la masa (FSM)**

En física nuclear, la fórmula de Weizsäcker permite estimar la masa y la energía de enlace $B$ de un núcleo atómico a partir de su número másico $A$ (protones + neutrones) y su número atómico $Z$ (protones). Es semiempírica porque combina argumentos teóricos con constantes ajustadas experimentalmente.

La energía de enlace $B$ representa la energía necesaria para desintegrar el núcleo. A mayor $B/A$ (energía por nucleón), más estable es el núcleo.

### La Fórmula
$$B(A,Z) = a_1 A - a_2 A^{2/3} - a_3 \frac{Z^2}{A^{1/3}} - a_4 \frac{(A-2Z)^2}{A} + \frac{a_5}{A^{1/2}}$$

Donde en MeV:
$a_1 = 15.8$, $a_2 = 18.3$, $a_3 = 0.714$, $a_4 = 23.2$

Y el término de apareamiento $a_5$:
* $a_5 = 0$ si $A$ es impar
* $a_5 = 12.0$ si $A$ y $Z$ son pares
* $a_5 = -12.0$ si $A$ es par y $Z$ es impar

Significado de los términos:
1. Volumen ($a_1 A$): atracción nuclear
2. Superficie ($-a_2 A^{2/3}$): corrección por nucleones en la superficie
3. Coulomb ($-a_3 Z^2 / A^{1/3}$): repulsión entre protones
4. Asimetría ($-a_4 (A-2Z)^2 / A$): penalización si N != Z
5. Apareamiento ($a_5 / A^{1/2}$): estabilidad extra si los nucleones están apareados


**a)** Escribe un programa que tome como entrada los valores de A y Z, e imprima la energ ́ıa de enlace B para el atomo correspondiente. Usa tu programa para encontrar  ́ la energ ́ıa de enlace de un atomo con  ́$A = 58$ y $Z = 28$ (Hint: La respuesta correcta es alrededor de los $490 MeV$).**

In [ ]:
def energia_enlace(A, Z):  # definimos la función. A es el numero masico (protones + neutrones) y Z es el numero atomico
    # Constantes en MeV
    a1 = 15.8    #termino de volumen
    a2 = 18.3  #termino de superficie
    a3 = 0.714  #termino de repulsion Coulombiana
    a4 = 23.2   #termino de asimetroa

    # Termino de a5
    if A % 2 == 1:  # A impar
        a5 = 0.0
    elif Z % 2 == 0:  # A par y Z par
        a5 = 12.0
    else:  # A par y Z impar
        a5 = -12.0

    B = (a1 * A  # formula completa. B es la Energia de enlace.
         - a2 * A**(2/3)  #la superficie
         - a3 * Z**2 / A**(1/3)  #es la repulsion electrica entre protones.
         - a4 * (A - 2*Z)**2 / A  #es la asimetria (diferencia entre neutrones y protones).
         + a5 / A**0.5)   # es la raiz cuadrada de A
    
    return B  #La funcion regresa el valor calculado de B para  usar afuera

# Programa que toma entrada
# #Aqui ya fuera de la funcion,se pide al usuario los datos. input() lee lo que se escribe y int() lo convierte a numero entero.
A = int(input("Introduce A (número másico): "))  
Z = int(input("Introduce Z (número atómico): "))

B = energia_enlace(A, Z) #Llamas a la funcion con los valores que tenemos
print(f"\nPara A={A}, Z={Z}:")
print(f"Energía de enlace B = {B:.2f} MeV")  #Imprime el resultado. El {B:.2f} significa "imprime B con solo 2 decimales"





Para A=58, Z=28:
Energía de enlace B = 497.56 MeV


**b)** Modifica el programa del inciso anterior, para escribir una segunda versión que imprima no la energía de enlace total B, sino la energía de unión por nucleón, que es B/A.

In [ ]:
def energia_enlace(A, Z):
    # Constantes en MeV
    a1 = 15.8
    a2 = 18.3
    a3 = 0.714
    a4 = 23.2

    # Termino de a5
    if A % 2 == 1:  # A impar
        a5 = 0.0
    elif Z % 2 == 0:  # A par y Z par
        a5 = 12.0
    else:  # A par y Z impar
        a5 = -12.0

    B = (a1 * A 
         - a2 * A**(2/3) 
         - a3 * Z**2 / A**(1/3) 
         - a4 * (A - 2*Z)**2 / A 
         + a5 / A**0.5)
    
    return B

# Programa que toma entrada
A = int(input("Introduce A (numero masico): "))
Z = int(input("Introduce Z (numero atomico): "))

B = energia_enlace(A, Z)
print(f"\nPara A={A}, Z={Z}:")
print(f"B/A = {B/A:.2f} MeV/nucleon") 
#B/A es la energia por nucleon, la que usasamos para saber que nucleo es mas estable. :.2f es para que lo deje con 2 decimales.




Para A=58, Z=28:
B/A = 8.58 MeV/nucleón


**c)** Escribe una tercera version del programa para que tome como entrada solo un valor  ́
del numero at  ́ omico  ́ Z y luego pase por todos los valores de $A$ desde $A = Z$ hasta
$A = 3Z$, para encontrar el que tiene la mayor energ ́ıa de enlace por nucleon. Este es  ́
el nucleo m  ́ as estable con el n  ́ umero at  ́ omico dado. Haz que tu programa imprima el  ́
valor de $A$ para este nucleo m  ́ as estable y el valor de la energ  ́  ́ıa de enlace por nucleon.

In [ ]:
def energia_enlace(A, Z):
    # Constantes en MeV
    a1 = 15.8
    a2 = 18.3
    a3 = 0.714
    a4 = 23.2

    # Termino de a5
    if A % 2 == 1:  # A impar
        a5 = 0.0
    elif Z % 2 == 0:  # A par y Z par
        a5 = 12.0
    else:  # A par y Z impar
        a5 = -12.0

    B = (a1 * A 
         - a2 * A**(2/3) 
         - a3 * Z**2 / A**(1/3) 
         - a4 * (A - 2*Z)**2 / A 
         + a5 / A**0.5)
    
    return B


Z = int(input("Introduce Z (numero atomico): ").strip())

max_B_por_A = -1e99
mejor_A = Z
mejor_B = 0.0

print(f"\nBuscando para Z={Z} desde A={Z} hasta A={3*Z}...\n")

for A in range(Z, 3*Z + 1):  #un nucleo no puede tener menos nucleones que protones, por eso empieza en A=Z 
    # en Python va de Z hasta 3Z-1 si no le ponemos +1. Por eso se pone +1, para incluir el 3Z
    B = energia_enlace(A, Z)  # calcula B total para este A con Z fijo
    B_por_A = B / A   # B/A = energia por nucleon, mide estabilidad

    if B_por_A > max_B_por_A:  #si este B/A es mayor que el maximo guardado

        max_B_por_A = B_por_A # actualiza el maximo B/A
        mejor_A = A   # guarda que A dio ese maximo
        mejor_B = B   # guarda su B total para no recalcular

print(f"Para Z={Z}:")
print(f"El nucleo mas estable es A = {mejor_A}")
print(f"B/A maximo = {max_B_por_A:.2f} MeV/nucleon")
print(f"B total = {mejor_B:.2f} MeV")


Buscando para Z=28 desde A=28 hasta A=84...

Para Z=28:
El nucleo mas estable es A = 62
B/A maximo = 8.70 MeV/nucleon
B total = 539.55 MeV


**d)** Finalmente, escribe una cuarta version del programa que, en lugar de tomar  ́ ́$ Z$ como
entrada, se ejecute a traves de todos los valores de  ́$ Z$ de 1 a 100 e imprima el valor
mas estable de  ́$ A$ para cada uno. ¿A que valor de  ́$ Z$ se produce la energ ́ıa de enlace

maxima por nucle  ́ on? (La respuesta correcta, en la vida real, es  ́$ Z = 28$, que corresponde al nıquel).

In [ ]:
def energia_enlace(A, Z):
    a1, a2, a3, a4 = 15.8, 18.3, 0.714, 23.2  # coeficientes en MeV
    if A % 2 == 1: a5 = 0.0      # A impar -> 0
    elif Z % 2 == 0: a5 = 12.0   # par-par -> +12 mas estable
    else: a5 = -12.0             # par-impar -> -12 menos estable
    B = a1*A - a2*A**(2/3) - a3*Z**2/A**(1/3) - a4*(A-2*Z)**2/A + a5/A**0.5
    return B  # B total


Z_max_global = 0        # Z donde esta el maximo global
A_max_global = 0        # A de ese Z maximo
BA_max_global = -1e99   # B/A maximo global

for Z in range(1, 101):  # recorre todos los Z de 1 a 100
    max_BA = -1e99       # maximo B/A para este Z
    mejor_A = Z          # A que da ese maximo para este Z
    mejor_B = 0.0        # B total de ese mejor A

    for A in range(Z, 3*Z + 1):  # para cada Z prueba A=Z hasta 3Z
        B = energia_enlace(A, Z) # B total de este (A,Z)
        BA = B / A               # B/A = estabilidad de este nucleo

        if BA > max_BA:          # si este es mas estable que el anterior
            max_BA = BA          # actualiza maximo de este Z
            mejor_A = A          # guarda A
            mejor_B = B          # guarda B

    print(f"Z={Z:3d} -> A mas estable={mejor_A:3d} con B/A={max_BA:.2f}") # imprime el mejor de este Z

    if max_BA > BA_max_global:    # compara con el maximo global de todos los Z
        BA_max_global = max_BA   # actualiza maximo global
        Z_max_global = Z         # guarda Z del maximo global
        A_max_global = mejor_A   # guarda A del maximo global

print(f"\nMaximo global en Z={Z_max_global}, A={A_max_global} con B/A={BA_max_global:.2f} MeV/nucleon")


Z=  1 -> A mas estable=  3 con B/A=0.37
Z=  2 -> A mas estable=  4 con B/A=5.32
Z=  3 -> A mas estable=  7 con B/A=5.28
Z=  4 -> A mas estable=  8 con B/A=6.47
Z=  5 -> A mas estable= 11 con B/A=6.65
Z=  6 -> A mas estable= 14 con B/A=7.20
Z=  7 -> A mas estable= 15 con B/A=7.33
Z=  8 -> A mas estable= 18 con B/A=7.72
Z=  9 -> A mas estable= 19 con B/A=7.74
Z= 10 -> A mas estable= 22 con B/A=8.04
Z= 11 -> A mas estable= 25 con B/A=8.03
Z= 12 -> A mas estable= 26 con B/A=8.24
Z= 13 -> A mas estable= 29 con B/A=8.24
Z= 14 -> A mas estable= 30 con B/A=8.38
Z= 15 -> A mas estable= 33 con B/A=8.39
Z= 16 -> A mas estable= 36 con B/A=8.49
Z= 17 -> A mas estable= 37 con B/A=8.48
Z= 18 -> A mas estable= 40 con B/A=8.57
Z= 19 -> A mas estable= 43 con B/A=8.55
Z= 20 -> A mas estable= 44 con B/A=8.63
Z= 21 -> A mas estable= 47 con B/A=8.61
Z= 22 -> A mas estable= 48 con B/A=8.66
Z= 23 -> A mas estable= 51 con B/A=8.65
Z= 24 -> A mas estable= 54 con B/A=8.69
Z= 25 -> A mas estable= 55 con B/A=8.66


## 3. Coeficientes binomiales

El coeficiente binomial $\binom{n}{k}$ es un número entero igual a:

$$
\binom{n}{k} = \frac{n!}{k!(n-k)!} = \frac{n \times (n-1) \times (n-2) \times \dots \times (n-k+1)}{1 \times 2 \times \dots \times k}
$$

donde $k \geq 1$, o bien 

$$
\binom{n}{0} = 1
$$ 

cuando $k = 0$.

**a)** Utiliza esta fórmula para escribir una función llamada `binomial(n,k)` (o como tú quieras) que calcule el coeficiente binomial para un n y k dados. Asegúrate de que tu función devuelva la respuesta en forma de un número entero (no flotante) y proporcione el valor correcto de 1 para el caso en que k = 0.

In [ ]:
def binomial(n, k): # define funcion n=n total, k=cuantos se toman
   
    if k == 0: # por definicion (n sobre 0) = 1
        return 1
    if k < 0 or k > n: # si k es invalido no existe, devuelve 0
        return 0
    if k > n - k: # truco: simetria (n,k) = (n, n-k) , es solo para optimizar, ahorra ciclos.
        k = n - k # 100 choose 90 = 100 choose 10, hace menos iteraciones
    
    resultado = 1 # acumulador, empieza en 1 por multiplicacion
    # hace n * (n-1) * ... / 1*2*... sin usar factoriales (evita numeros gigantes)
    for i in range(1, k + 1): # i va de 1 hasta k inclusive
        resultado = resultado * (n - i + 1) // i # // es division entera,  asegura int , n - i + 1 : es el numerador
    
    return resultado  # siempre entero, no flotante 

print(binomial(100, 0))  # prueba k=0 debe dar 1
print(binomial(4, 2))    # prueba 4 sobre 2 = 6

1
6


b) Usando tu función, escribe un programa que imprima las primeras 20 líneas del "triángulo de Pascal". La n-ésima línea del triángulo de Pascal contiene $n+1$ números, que son los coeficientes $\binom{n}{0}$, $\binom{n}{1}$, y así sucesivamente hasta $\binom{n}{n}$. De tal manera que las primeras líneas son:

1                                                                                                                                         
1 1                                                                                                                                       
1 2 1                                                                                                                                     
1 3 3 1                                                                                                                                   
1 4 6 4 1                                                                                                                                 

In [18]:
def binomial(n, k): # n fila, k columna
    if k == 0: # caso base  (n,0)=1
        return 1
    if k < 0 or k > n: # k fuera de rango no existe
        return 0
    if k > n - k: # simetria (n,k)=(n,n-k) para hacer menos vueltas
        k = n - k
    
    resultado = 1 # acumulador de la multiplicacion
    for i in range(1, k + 1): # i=1 hasta k
        resultado = resultado * (n - i + 1) // i # // division entera, asegura int
    return resultado # devuelve entero, no float

# imprime 20 lineas del triangulo de Pascal
for n in range(20): # n=0 a 19 a 20 lineas, la n-esima tiene n+1 numeros
    for k in range(n + 1): # k=0 hasta n -> coeficientes (n,0) a (n,n)
        print(binomial(n, k), end=' ') # end=' ' pone espacio, no salto, evita 1111
    print() # salto de linea al terminar la fila n

1 
1 1 
1 2 1 
1 3 3 1 
1 4 6 4 1 
1 5 10 10 5 1 
1 6 15 20 15 6 1 
1 7 21 35 35 21 7 1 
1 8 28 56 70 56 28 8 1 
1 9 36 84 126 126 84 36 9 1 
1 10 45 120 210 252 210 120 45 10 1 
1 11 55 165 330 462 462 330 165 55 11 1 
1 12 66 220 495 792 924 792 495 220 66 12 1 
1 13 78 286 715 1287 1716 1716 1287 715 286 78 13 1 
1 14 91 364 1001 2002 3003 3432 3003 2002 1001 364 91 14 1 
1 15 105 455 1365 3003 5005 6435 6435 5005 3003 1365 455 105 15 1 
1 16 120 560 1820 4368 8008 11440 12870 11440 8008 4368 1820 560 120 16 1 
1 17 136 680 2380 6188 12376 19448 24310 24310 19448 12376 6188 2380 680 136 17 1 
1 18 153 816 3060 8568 18564 31824 43758 48620 43758 31824 18564 8568 3060 816 153 18 1 
1 19 171 969 3876 11628 27132 50388 75582 92378 92378 75582 50388 27132 11628 3876 969 171 19 1 


**triangulo centrado**

In [19]:
def binomial(n, k): # n fila, k columna
    if k == 0: # caso (n,0)=1
        return 1
    if k < 0 or k > n: # k fuera de rango no existe
        return 0
    if k > n - k: # simetria (n,k)=(n,n-k) para hacer menos vueltas
        k = n - k
    
    resultado = 1 # acumulador de la multiplicacion
    for i in range(1, k + 1): # i=1 hasta k
        resultado = resultado * (n - i + 1) // i # // division entera, asegura int
    return resultado # devuelve entero, no float

# triangulo centrado
for n in range(20):
    linea = ' '.join(str(binomial(n, k)) for k in range(n+1))
    print(linea.center(60)) # .center centra el texto

                             1                              
                            1 1                             
                           1 2 1                            
                          1 3 3 1                           
                         1 4 6 4 1                          
                       1 5 10 10 5 1                        
                      1 6 15 20 15 6 1                      
                    1 7 21 35 35 21 7 1                     
                   1 8 28 56 70 56 28 8 1                   
                1 9 36 84 126 126 84 36 9 1                 
            1 10 45 120 210 252 210 120 45 10 1             
          1 11 55 165 330 462 462 330 165 55 11 1           
        1 12 66 220 495 792 924 792 495 220 66 12 1         
    1 13 78 286 715 1287 1716 1716 1287 715 286 78 13 1     
 1 14 91 364 1001 2002 3003 3432 3003 2002 1001 364 91 14 1 
1 15 105 455 1365 3003 5005 6435 6435 5005 3003 1365 455 105 15 1
1 16 120 560 1820 4

**c)** La probabilidad de que para una moneda no sesgada, lanzada $n$ veces, salga águila $k$ veces es:

$$ p(k|n) = \frac{\binom{n}{k}}{2^n} $$

Escribe un programa para calcular:

1) la probabilidad total de que una moneda lanzada 100 veces, salga águila exactamente 60 veces y
2) la probabilidad de que salga águila 60 veces o más.

In [ ]:
def binomial(n, k): # misma funcion del inciso a
    if k == 0:
        return 1
    if k < 0 or k > n:
        return 0
    if k > n - k: # truco para menos vueltas
        k = n - k
    resultado = 1
    for i in range(1, k + 1): # calcula n*(n-1)/1*2...
        resultado = resultado * (n - i + 1) // i # // entero
    return resultado

# c) p(k|n) = (n sobre k) / 2^n

n = 100 # lanzamientos totales
k = 60  # aguilas que queremos

#  Probabilidad exactamente 60
comb = binomial(n, k) # (100 sobre 60)
p_exacta = comb / (2**n) # divide entre 2^n aplicamos la formula ,   2^100 total de combinaciones posibles de una moneda, potencia.
print(f"1) P(k=60|n=100) = {p_exacta}") 

#  Probabilidad 60 o mas = suma de k=60 hasta 100
p_60_o_mas = 0.0 # acumulador de probabilidad
for k_actual in range(60, n + 1): # recorre 60,61,...100
    p_60_o_mas += binomial(n, k_actual) / (2**n) # suma p(k|n) de cada k

print(f"2) P(k>=60|n=100) = {p_60_o_mas}") 

1) P(k=60|n=100) = 0.010843866711637987
2) P(k>=60|n=100) = 0.028443966820490392


## 4. Números primos

Es posible escribir un programa bastante rápido para calcular números primos siguiendo las siguientes observaciones:



**a)** Un número $n$ es primo, si no tiene factores primos menores que $n$. Por lo tanto, sólo necesitamos comprobar si es divisible por otros números primos.

In [22]:
def es_primo(n, primos_previos):
    # n: numero a probar
    # primos_previos: lista de primos menores que n
    for p in primos_previos: # solo pruebamos con primos, no con todos
        if p * p > n: # truco: si p^2 > n, ya no hay factores
            break
        if n % p == 0: # % es residuo. Si es 0, es divisible
            return False # tiene divisor primo entoncs no es primo
    return True # no fue divisible por ningun primo entones si es primo

def calcular_primos(hasta):
    # hasta: hasta que numero buscar
    primos = [2] # 2 es el primer primo
    for n in range(3, hasta + 1, 2): # solo impares, los pares no son primos 
        if es_primo(n, primos): # usamos la primera observacion 
            primos.append(n) # aqui lo agrega a la lista de primos
    return primos

# Prueba: primeros primos
primos_100 = calcular_primos(100)
print(primos_100)
print(f"Hay {len(primos_100)} primos hasta 100")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]
Hay 25 primos hasta 100


**b)** Si un número $n$ no es primo y tiene un factor $r$, entonces $n = r s$, donde $s$ también es un factor. Si $r \geq \sqrt{n}$ entonces $n = r s \geq \sqrt{n} s$, lo que implica que $s \leq \sqrt{n}$. Es decir, cualquier no primo debe tener factores, y por lo tanto también factores primos, menores o iguales a $\sqrt{n}$. Por lo tanto, para determinar si un número es primo, tenemos que verificar sus factores primos solo hasta $\sqrt{n}$; incluso si no hay ninguno, entonces el número es primo.

In [25]:
import math

def es_primo_b(n, primos_previos):
    # n: numero a probar
    # primos_previos: primos menores que n
    limite = math.isqrt(n) # limite = sqrt(n)
    for p in primos_previos:
        if p > limite: # b) si p ya paso la raiz, paramos
            break
        if n % p == 0:
            return False
    return True

def calcular_primos_b(hasta):
    primos = [2]
    for n in range(3, hasta + 1, 2): # solo impares
        if es_primo_b(n, primos):
            primos.append(n)
    return primos

print(calcular_primos_b(100))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]


**c)** Si encontramos un factor primo menor que $\sqrt{n}$ entonces sabemos que el número no es primo y, por lo tanto, no hay necesidad de realizar más comprobaciones; podemos abandonar este número y pasar al siguiente.

In [26]:
import math

def es_primo_c(n, primos_previos):
    # n: numero a probar
    limite = math.isqrt(n) # hasta raiz de n
    for p in primos_previos:
        if p > limite: # si ya paso la raiz
            break
        if n % p == 0: # c) encontramos factor primo < raiz
            return False # -> no es primo, abandonar y pasar al siguiente
    return True

def calcular_primos_c(hasta):
    primos = [2]
    for n in range(3, hasta + 1, 2):
        if es_primo_c(n, primos): # si no se abandono, es primo
            primos.append(n)
    return primos

print(calcular_primos_c(100))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]


Escribe un programa que encuentra todos los números primos hasta 10000 (diez mil). Crea una lista para almacenar los números primos, que comience con solo un número primo (el 2). Luego, para cada número n del 3 al 10000, verifica si el número es divisible por alguno de los números primos de la lista hasta √n. En cuanto encuentres un factor primo, puedes dejar de comprobar el resto de números (pues ya sabrías que n no es primo). Si no encuentras factores primos iguales o menores a √n, entonces n es primo y habría que agregarlo a la lista. Puedes imprimir la lista completa al final del programa, o bien, ir imprimiendo los números individualmente a medida que los vayas encontrando.

In [ ]:
import math

def primos_hasta_10000():
    primos = [2] # lista que comienza solo con el 2

    # para cada n del 3 al 10000
    for n in range(3, 10001):
        limite = math.isqrt(n) # hasta √n
        es_primo = True

        # verifica si es divisible por primos de la lista hasta √n
        for p in primos:
            if p > limite: #  solo hasta √n
                break
            if n % p == 0: # encontramos factor primo
                es_primo = False
                break # deja de comprobar el resto, porque se sabe que no es primo

        # si no encontro factores <= √n, es primo
        if es_primo:
            primos.append(n)
            print(n) # imprime individualmente a medida que los encuentra

    return primos


lista_primos = primos_hasta_10000()
print("\n--- Lista completa al final ---")
print(lista_primos)
print(f"\nTotal de primos hasta 10000: {len(lista_primos)}")

3
5
7
11
13
17
19
23
29
31
37
41
43
47
53
59
61
67
71
73
79
83
89
97
101
103
107
109
113
127
131
137
139
149
151
157
163
167
173
179
181
191
193
197
199
211
223
227
229
233
239
241
251
257
263
269
271
277
281
283
293
307
311
313
317
331
337
347
349
353
359
367
373
379
383
389
397
401
409
419
421
431
433
439
443
449
457
461
463
467
479
487
491
499
503
509
521
523
541
547
557
563
569
571
577
587
593
599
601
607
613
617
619
631
641
643
647
653
659
661
673
677
683
691
701
709
719
727
733
739
743
751
757
761
769
773
787
797
809
811
821
823
827
829
839
853
857
859
863
877
881
883
887
907
911
919
929
937
941
947
953
967
971
977
983
991
997
1009
1013
1019
1021
1031
1033
1039
1049
1051
1061
1063
1069
1087
1091
1093
1097
1103
1109
1117
1123
1129
1151
1153
1163
1171
1181
1187
1193
1201
1213
1217
1223
1229
1231
1237
1249
1259
1277
1279
1283
1289
1291
1297
1301
1303
1307
1319
1321
1327
1361
1367
1373
1381
1399
1409
1423
1427
1429
1433
1439
1447
1451
1453
1459
1471
1481
1483
1487
1489
1493
1499
1511

## 5. Recursión



**a)** Anteriormente, en clase encontramos los números de Catalán Cₙ. Si hacemos una pequeña modificación, la definición de Cₙ se puede reescribir de la siguiente forma:

       Cₙ = { 1                          si n = 0,
            { (4n - 2)/(n + 1) * Cₙ₋₁    si n > 0.

Escribe una función, usando recursividad, que calcule Cₙ y utilízala para calcular e imprimir C₁₀₀.

In [ ]:
def catalan(n):  #  define funcion recursiva.
    if n == 0: # caso base: C0 = 1 por definicion, condicion de paro. Sin esto la recursion nunca termina.
        return 1
    else: # caso recursivo: Cn = (4n-2)/(n+1) * Cn-1
        return catalan(n-1) * (4*n - 2) // (n + 1) # llamada recursiva.  para obtener el anterior
    #aplicamos la formula . Uso // en lugar de / porque C_n siempre es entero y así evitamos decimales.

# se calcula e imprime C100
c100 = catalan(100) # lanzamos la cadena: C100 a C99 -a C98 ... a C0 y se va regresando multiplicando.
print(c100) # imprime el resultado. 
print(f"\nC_100 tiene {len(str(c100))} digitos")

896519947090131496687170070074100632420837521538745909320

C_100 tiene 57 digitos


**b)** Euclides demostró que el máximo común divisor g(m,n) de dos enteros no negativos m y n satisface que:

       g(m,n) = { m                si n = 0,
                { g(n, m mod n)    si n > 0.

Usando esta fórmula escribe una función g(m,n) que emplee recursividad para calcular el máximo común divisor de m y n. Usa tu función para calcular e imprimir el máximo común divisor de 108 y 192.

In [32]:
def g(m, n): # se define la funcion g(m,n)
    # m, n: enteros no negativos
    if n == 0:  # CASO BASE: si n es 0, el mcd es m,  Es la condicion de paro de la recursion. 
        return m
    else:       # CASO RECURSIVO: g(m,n) = g(n, m mod n)
        return g(n, m % n) # m % n es el residuo de dividir m entre n, paso recursivo. La funcion se llama a si misma pero intercambiando lugares:
        #el nuevo m es el viejo n, y el nuevo n es el residuo. Cada llamada hace los numeros mas pequeños.

# Usamos la funcion para 108 y 192
resultado = g(108, 192)
print(f"mcd(108, 192) = {resultado}")

mcd(108, 192) = 12
